# VRC Compact59 — Colab GPU 学習

**「ランタイム」→「ランタイムのタイプを変更」で GPU を選択し、「すべてのセルを実行」してください。**

入力は `VRC_Compact59_Colab_7fe4fac1.zip`。完了時は **`VRC_Compact59_Colab_Result_7fe4fac1.zip`**、失敗時は **`VRC_Compact59_Colab_Diagnostic_7fe4fac1.zip`** がダウンロードされます。生成された名前のまま元のチャットへ添付してください。

失敗時も保存済み checkpoint / ONNX と具体的な失敗理由を回収します。初期化直後・途中のモデルを完成モデルとしては扱いません。HF_TOKEN・GitHub認証・Driveマウントは不要です。


In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import hashlib, io, json, os, stat, subprocess, sys, zipfile

KIT_NAME = "VRC_Compact59_Colab_7fe4fac1.zip"
ROOT_NAME = "VRC_Compact59_Colab_7fe4fac1"
CONTENT = Path("/content")
ROOT = CONTENT / ROOT_NAME
os.chdir(CONTENT)
print("入力ZIPを選択してください:", KIT_NAME)
uploaded = files.upload()
if len(uploaded) != 1 or KIT_NAME not in uploaded:
    raise ValueError("指定名のZIPを1個だけアップロードしてください: " + KIT_NAME)
payload = uploaded[KIT_NAME]
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    members = archive.infolist()
    if len(members) > 100 or sum(m.file_size for m in members) > 128 * 1024 * 1024:
        raise ValueError("学習キットのファイル数またはサイズが上限を超えています")
    for member in members:
        path = PurePosixPath(member.filename)
        if path.is_absolute() or ".." in path.parts or not path.parts or path.parts[0] != ROOT_NAME:
            raise ValueError("学習キット外への展開は許可されていません")
        if stat.S_ISLNK(member.external_attr >> 16):
            raise ValueError("シンボリックリンクは受け付けません")
    bundle = json.loads(archive.read(ROOT_NAME + "/bundle.json"))
    if bundle.get("handoffId") != "7fe4fac1" or bundle.get("privateInputsIncluded") is not False:
        raise ValueError("このチャット向けの学習キットではありません")
    expected = {ROOT_NAME + "/" + entry["path"]: entry for entry in bundle["files"]}
    actual = {m.filename for m in members if not m.is_dir()} - {ROOT_NAME + "/bundle.json"}
    if actual != set(expected):
        raise ValueError("キットのファイル一覧が一致しません")
    for name, entry in expected.items():
        data = archive.read(name)
        if len(data) != entry["bytes"] or hashlib.sha256(data).hexdigest() != entry["sha256"]:
            raise ValueError("キットのSHA-256が一致しません: " + name)
    archive.extractall(CONTENT)
print("入力確認完了:", hashlib.sha256(payload).hexdigest())
del uploaded, payload


## 依存関係

Colabに割り当てられたCUDA版PyTorchを使います。追加ライブラリは専用ディレクトリへ入れ、学習用の子プロセスだけで読み込みます。`venv` / `ensurepip` は使用せず、ノートブック本体のパッケージを置き換えません。


In [ ]:
import runpy

PYTHON = sys.executable
RUN_ENV = os.environ.copy()

def collect_result():
    packaged = subprocess.run(
        [PYTHON, str(ROOT / "colab" / "package_result.py"), "--root", str(ROOT),
         "--destination", str(CONTENT)],
        env=RUN_ENV, check=True, text=True, capture_output=True)
    receipt = json.loads(packaged.stdout.strip().splitlines()[-1])
    print("実行状態:", receipt["status"])
    print("ONNX:", receipt["onnxFiles"] or "生成されていません")
    print("返却ファイル:", receipt["fileName"])
    files.download(str(CONTENT / receipt["fileName"]))
    return receipt

try:
    RUN_ENV = runpy.run_path(str(ROOT / "colab" / "bootstrap.py"))["install"](ROOT)
except Exception as error:
    (ROOT / "output").mkdir(exist_ok=True)
    safe_error = runpy.run_path(str(ROOT / "colab" / "run.py"))["safe_error"]
    failure = {"schema": "vrc-colab-run-v2", "handoffId": "7fe4fac1", "status": "failed",
               "privateInputsIncluded": False, "sourceCommit": bundle["sourceCommit"],
               "stages": [{"name": "dependency_setup", "status": "failed"}],
               "errorType": type(error).__name__, "errorMessage": safe_error(str(error))}
    (ROOT / "output" / "run-status.json").write_text(
        json.dumps(failure, ensure_ascii=False, indent=2), encoding="utf-8")
    collect_result()
    raise


## GPU学習と評価

2候補を4000 optimizer stepずつ学習します。`BATCH_SIZE=32` は1更新あたり32枚、`MICRO_BATCH_SIZE=8` は一度にGPUへ載せる8枚です。この設定は4回の勾配蓄積になります。GPUメモリを守るため、BATCH_SIZEは2〜256、MICRO_BATCH_SIZEは1〜16を受け付けます。大きなBATCH_SIZEは同じstep数でも学習時間を増やします。

初期化直後・1 step目・100 stepごとにモデルを保存します。学習768画像、候補選択128画像、最終test192画像。分割IDを学習開始前に保存するため、GPU学習が失敗しても独立したINT8実験は続行できます。

このセルで結果ZIPを作成しダウンロードします。失敗した場合は診断ZIPをダウンロードしてからセルをエラーにし、成功したようには表示しません。


In [ ]:
TRAIN_STEPS = 4000
BATCH_SIZE = 32
MICRO_BATCH_SIZE = 8
training = None
try:
    training = subprocess.run(
        [PYTHON, str(ROOT / "colab" / "run.py"), "--root", str(ROOT),
         "--steps", str(TRAIN_STEPS), "--batch-size", str(BATCH_SIZE),
         "--micro-batch-size", str(MICRO_BATCH_SIZE)], env=RUN_ENV, check=False)
finally:
    if (ROOT / "output" / "run-status.json").is_file():
        result_receipt = collect_result()
if training.returncode != 0 or result_receipt["status"] != "completed":
    raise RuntimeError("学習は未完了です。ダウンロード済みの Diagnostic ZIP を元のチャットへ添付してください。")


## 返却内容の確認

上のセルがダウンロードしたZIPを、元のChatGPTチャットへ名前を変えずに添付してください。Resultは全工程完了、Diagnosticは未完了です。Diagnosticに含まれる途中の重みは回収用で、精度の合格や公開を意味しません。

画像、動画、切り抜き、特徴キャッシュは返却ZIPに含めません。


In [ ]:
print("返却ファイル:", result_receipt["fileName"])
print("ONNXを含む:", result_receipt["containsOnnx"])
print("実行状態:", result_receipt["status"])
